# Lesson 4.8: Polars and DuckDB

**Analyst:** Meron Welderufael
**Date:** 11/10/2026
**Notebook purpose:** Same per-tier FinFlow analysis in three tools.
Pandas (familiar), Polars (the faster DataFrame), DuckDB (SQL inside
Python). Benchmark each. Build the foundation for using the right tool
on the next engagement.

In [1]:
import pandas as pd
import polars as pl
import duckdb
import time


df_pandas = pd.read_csv('finflow-enriched-200k.csv',
                       parse_dates = ['signup_date', 'last_login_date', 'churn_date'])
print(f'shape: {df_pandas.shape}')

shape: (200000, 22)


## Implementation 1: Pandas (reference).

In [11]:
start = time.time()

df_pandas['signup_cohort'] = df_pandas['signup_date'].dt.to_period('M')

def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

cohort_churn_pandas = df_pandas.groupby('signup_cohort').agg(
    customer_count=('customer_id', 'count'),
    churn_rate_pct = ('active', churn_rate_pct),
    avg_monthly_fee = ('monthly_fee', 'mean')   
).round(2)

pandas_time = time.time() - start

print(f"Pandas time: {pandas_time:.4f}s")
print(cohort_churn_pandas.head(10))
print('---rows omitted---')
print(cohort_churn_pandas.tail(5))

Pandas time: 0.0830s
               customer_count  churn_rate_pct  avg_monthly_fee
signup_cohort                                                 
2024-10                    60           11.67            41.86
2024-11                   774           19.77            37.27
2024-12                  1779           19.00            33.37
2025-01                  2645           18.83            35.35
2025-02                  3300           19.73            35.81
2025-03                  4457           21.05            35.25
2025-04                  5178           20.01            35.42
2025-05                  6359           20.49            35.29
2025-06                  7145           20.20            34.87
2025-07                  8195           19.77            34.72
---rows omitted---
               customer_count  churn_rate_pct  avg_monthly_fee
signup_cohort                                                 
2026-06                 11794           20.21            35.07
2026-07        

## Implementation 2: Polars.

In [10]:
df_polars = pl.read_csv('finflow-enriched-200k.csv',
                       try_parse_dates= True)

start = time.time()
df_polars = df_polars.with_columns(
    pl.col('signup_date').dt.truncate('1mo').alias('signup_cohort_polars')
)

cohort_churn_polars = df_polars.group_by('signup_cohort_polars').agg([
    pl.col('customer_id').count().alias('customer_count'),
    (pl.col('active').not_().sum()/pl.col('active').count()*100).alias('churn_rate_pct'),
    pl.col('monthly_fee').mean().alias('avg_monthly_fee')
]).sort('signup_cohort_polars')

polars_time = time.time() - start
print(f"Polars time: {polars_time:.4f}s")
print(cohort_churn_polars.head(10))
print('---rows omitted---')
print(cohort_churn_polars.tail(5))

Polars time: 0.0359s
shape: (10, 4)
┌──────────────────────┬────────────────┬────────────────┬─────────────────┐
│ signup_cohort_polars ┆ customer_count ┆ churn_rate_pct ┆ avg_monthly_fee │
│ ---                  ┆ ---            ┆ ---            ┆ ---             │
│ date                 ┆ u32            ┆ f64            ┆ f64             │
╞══════════════════════╪════════════════╪════════════════╪═════════════════╡
│ 2024-10-01           ┆ 60             ┆ 11.666667      ┆ 41.858          │
│ 2024-11-01           ┆ 774            ┆ 19.767442      ┆ 37.270181       │
│ 2024-12-01           ┆ 1779           ┆ 18.999438      ┆ 33.365306       │
│ 2025-01-01           ┆ 2645           ┆ 18.827977      ┆ 35.348299       │
│ 2025-02-01           ┆ 3300           ┆ 19.727273      ┆ 35.810142       │
│ 2025-03-01           ┆ 4457           ┆ 21.045546      ┆ 35.249486       │
│ 2025-04-01           ┆ 5178           ┆ 20.007725      ┆ 35.416491       │
│ 2025-05-01           ┆ 6359           

## Implementation 3: DuckDB SQL.

In [14]:
start = time.time()

df_duck = df_pandas[["signup_date", "active", "monthly_fee"]]
cohort_churn_duckdb = duckdb.sql("""
     select 
         CAST(DATE_TRUNC('month', signup_date) AS DATE) as signup_cohort_duck,
         count(*) as customer_count,
         (sum(case when active=false then 1 else 0 end)*100.0/count(*)) as churn_rate_pct,
         avg(monthly_fee) as avg_monthly_fee
     from df_duck
     group by DATE_TRUNC('month', signup_date)
     order by DATE_TRUNC('month', signup_date)
""").df()

duckdb_time = time.time() - start
print(f"DuckDB time: {duckdb_time:.4f}s")
print(cohort_churn_duckdb.head(10))
print('---rows omitted---')
print(cohort_churn_duckdb.tail(5))


DuckDB time: 0.0623s
  signup_cohort_duck  customer_count  churn_rate_pct  avg_monthly_fee
0         2024-10-01              60       11.666667        41.858000
1         2024-11-01             774       19.767442        37.270181
2         2024-12-01            1779       18.999438        33.365306
3         2025-01-01            2645       18.827977        35.348299
4         2025-02-01            3300       19.727273        35.810142
5         2025-03-01            4457       21.045546        35.249486
6         2025-04-01            5178       20.007725        35.416491
7         2025-05-01            6359       20.490643        35.288231
8         2025-06-01            7145       20.195941        34.869514
9         2025-07-01            8195       19.768151        34.718140
---rows omitted---
   signup_cohort_duck  customer_count  churn_rate_pct  avg_monthly_fee
20         2026-06-01           11794       20.205189        35.069351
21         2026-07-01            9111       19.6

## Timing table

In [15]:
print(f"Pandas:  {pandas_time:.4f}s")
print(f"Polars:  {polars_time:.4f}s")
print(f"DuckDB:  {duckdb_time:.4f}s")
print(f"\nFastest is {min((pandas_time, 'Pandas'), (polars_time, 'Polars'), (duckdb_time, 'DuckDB'))[1]}")

Pandas:  0.0830s
Polars:  0.0359s
DuckDB:  0.0623s

Fastest is Polars


**Which implementation read most naturally to you? Why?**

Polars felt the most natural to me because its table structure is familiar, and it clearly displays each column's data type. This makes it easier to understand the data and work with it.

**Which gave the cleanest output (least manual cleanup needed)?**

I think both Polars and DuckDB produced clean outputs, but I prefer Polars because its syntax is straightforward and requires less manual cleanup.

**If this same analysis ran every Monday morning on a 20-million-row warehouse table, which tool would you reach for, and why?**

I would choose Polars because it is fast and its syntax is similar to SQL. However, if the analysis involved more complex joins and aggregations, I would consider DuckDB because it makes writing complex SQL queries easier.